# 03 — MIDAS (Weekly, EXOG + Macro + FRED/COT + Sentiment) — Python port

Python/`scipy` port of the R `03_midas.ipynb` (kept alongside). Weekly W-FRI silver log-return;
**base** = silver AR lags + 6 cross-asset weekly lags (EXOG); **MIDAS-native** monthly macro
(`cpi, fed_funds, ind_prod, m2`, 3 publication-availability lags) via Beta / exp-Almon / U-MIDAS.
Public-info rungs mirror `01_arima`: **FRED_daily** (`real_rates_chg, breakeven_chg, jobless_chg`,
summed→weekly) and **COT** (`cot_mm_net_pct, cot_comm_net_pct`, last→weekly), both `.shift(1)`.
Reuses `eval_utils`; writes `metrics_midas.csv` / `period_midas_weekly.csv` / `preds_midas_best_weekly.csv`
(+ RMSE-best siblings). Full §3a battery incl. always-up line, best-by-RMSE, and richer PT.


## Setup

In [ ]:
import sys, os
import numpy as np
import pandas as pd
from scipy.optimize import minimize
import matplotlib.pyplot as plt
sys.path.append('../../src')
from eval_utils import evaluate, period_metrics, diebold_mariano, pesaran_timmermann, oos_r2, PERIODS

DATA = '../../data/processed/'
RAW  = '../../data/raw/'


## 1. Load & aggregate to weekly

In [ ]:
train = pd.read_csv(DATA+'train.csv', index_col=0, parse_dates=True)
val   = pd.read_csv(DATA+'val.csv',   index_col=0, parse_dates=True)
test  = pd.read_csv(DATA+'test.csv',  index_col=0, parse_dates=True)

TARGET       = 'silver_return'
EXOG_RETURNS = ['gold_return','usd_return','copper_return','sp500_return','vix_return','oil_return']
EXOG_LEVELS  = ['gs_ratio_z']
FRED_DAILY   = ['real_rates_chg','breakeven_chg','jobless_chg']   # public-info group (mirrors 01_arima)
COT_COLS     = ['cot_mm_net_pct','cot_comm_net_pct']              # COT positioning

# Weekly W-FRI aggregation, matching 02_features / 01_arima:
#   returns + FRED Δ-changes -> sum ; levels (gs_ratio_z) + COT net-% -> last.
def to_weekly(df):
    summ = [c for c in [TARGET]+EXOG_RETURNS+FRED_DAILY if c in df.columns]
    last = [c for c in EXOG_LEVELS+COT_COLS         if c in df.columns]
    agg = {c:'sum' for c in summ}; agg.update({c:'last' for c in last})
    return df.resample('W-FRI').agg(agg)

train_w, val_w, test_w = to_weekly(train), to_weekly(val), to_weekly(test)
all_w   = pd.concat([train_w, val_w, test_w]).sort_index()
n_train = len(train_w) + len(val_w)
print(f'Weekly obs — train+val: {n_train}, test: {len(test_w)}')


## 2. EXOG base + public-info (FRED/COT) lags

In [ ]:
for k in (1,2,3):
    all_w[f'silver_lag{k}'] = all_w['silver_return'].shift(k)
for v in EXOG_RETURNS:
    all_w[v.replace('_return','_lag1')] = all_w[v].shift(1)
for v in EXOG_LEVELS:
    all_w[f'{v}_lag1'] = all_w[v].shift(1)
# FRED_daily + COT public-info groups: 1-week lag, warmup filled with 0 (matches 01_arima)
for v in FRED_DAILY + COT_COLS:
    if v in all_w.columns:
        all_w[f'{v}_lag1'] = all_w[v].shift(1).fillna(0.0)

EXOG_LAGS = [c for c in ['silver_lag1','silver_lag2','silver_lag3',
                         'gold_lag1','usd_lag1','copper_lag1','sp500_lag1','vix_lag1','oil_lag1']
             if c in all_w.columns]
GS_LAGS   = [f'{v}_lag1' for v in EXOG_LEVELS if f'{v}_lag1' in all_w.columns]
FRED_LAGS = [f'{v}_lag1' for v in FRED_DAILY  if f'{v}_lag1' in all_w.columns]
COT_LAGS  = [f'{v}_lag1' for v in COT_COLS    if f'{v}_lag1' in all_w.columns]
print('EXOG:', EXOG_LAGS); print('GS:', GS_LAGS, '| FRED:', FRED_LAGS, '| COT:', COT_LAGS)


## 3. Monthly macro lag matrices (MIDAS predictors)

In [ ]:
macro = pd.read_csv(RAW+'monthly_macro.csv', index_col=0, parse_dates=True)
MACRO_VARS = ['cpi','fed_funds','ind_prod','m2']
assert all(v in macro.columns for v in MACRO_VARS), 'monthly_macro.csv missing a column'
N_MACRO_LAGS    = 3
MACRO_AVAIL_LAG = {'cpi':46,'ind_prod':48,'fed_funds':35,'m2':30}

def build_weekly_macro_lags(weekly_dates, s, lag_days, n_lags=N_MACRO_LAGS):
    avail = s.index + pd.Timedelta(days=lag_days)
    vals  = s.values; obs = ~np.isnan(vals)
    out   = np.full((len(weekly_dates), n_lags), np.nan)
    for i, d in enumerate(weekly_dates):
        keep = obs & (avail < d - pd.Timedelta(days=6))
        past = vals[keep]
        if len(past) >= n_lags:
            out[i] = past[-n_lags:][::-1]
    return out

macro_lags = {v: build_weekly_macro_lags(all_w.index, macro[v], MACRO_AVAIL_LAG[v]) for v in MACRO_VARS}
print('Macro lag matrices:', {v: m.shape for v, m in macro_lags.items()})


### Look-ahead audit

In [ ]:
position_fri = all_w.index - pd.Timedelta(days=7)
for v in MACRO_VARS:
    avail = macro.index + pd.Timedelta(days=MACRO_AVAIL_LAG[v]); obs = macro[v].notna().values
    min_slack = np.inf
    for i, d in enumerate(all_w.index):
        keep = obs & (avail < d - pd.Timedelta(days=6))
        if keep.any():
            min_slack = min(min_slack, (position_fri[i] - avail[keep].max()).days)
    assert not (np.isfinite(min_slack) and min_slack < 0), f'look-ahead in {v}'
    print(f'  {v:11s} OK  (min slack {int(min_slack)} d)')
print('Macro look-ahead audit passed.')


## 4. Weekly sentiment (lagged 1 week)

In [ ]:
sent = pd.read_csv(DATA+'daily_sentiment.csv', index_col=0, parse_dates=True)
sent_cols = [c for c in ['reddit_sentiment','news_sentiment'] if c in sent.columns]
sentiment_available = len(sent_cols) > 0
aligned = sent[sent_cols].resample('W-FRI').mean().reindex(all_w.index)
if 'reddit_sentiment' in sent_cols:
    all_w['reddit_lag1'] = aligned['reddit_sentiment'].ffill().shift(1).fillna(0.0)
if 'news_sentiment' in sent_cols:
    all_w['news_lag1'] = aligned['news_sentiment'].ffill().shift(1).fillna(0.0)
print('Sentiment merged:', [c for c in ['reddit_lag1','news_lag1'] if c in all_w.columns])


## 5. MIDAS weight functions and fitters

In [ ]:
def nbeta_w(k, theta):
    th = np.maximum(theta, 0.1)
    x  = k / (k.max() + 1)
    w  = x**(th[0]-1) * (1-x)**(th[1]-1)
    w  = np.maximum(w, 1e-10)
    return w / w.sum()

def nealmon_w(k, theta):
    e = theta[0]*k + theta[1]*k**2
    e = e - e.max()
    w = np.exp(e)
    return w / w.sum()


In [ ]:
def fit_with_midas(y, base_X, macro_list, weight_fn=nbeta_w, start=None, lower=None):
    names = list(macro_list); nmac = len(names)
    klists = {v: np.arange(1, macro_list[v].shape[1]+1) for v in names}
    if start is None: start = np.tile([1.0,5.0], nmac)
    if lower is None: lower = np.full(2*nmac, 0.1)
    base = np.asarray(base_X, float); y = np.asarray(y, float)
    def design(theta):
        cols = [macro_list[v] @ weight_fn(klists[v], theta[2*j:2*j+2]) for j, v in enumerate(names)]
        return np.column_stack([np.ones(len(y)), base, np.column_stack(cols)])
    def obj(theta):
        X = design(theta)
        if not np.all(np.isfinite(X)): return 1e9
        b, *_ = np.linalg.lstsq(X, y, rcond=None)
        r = y - X @ b
        return float(r @ r)
    res = minimize(obj, start, method='L-BFGS-B', bounds=[(lo, None) for lo in lower], options={'maxiter':1000})
    theta = res.x
    weights = {v: weight_fn(klists[v], theta[2*j:2*j+2]) for j, v in enumerate(names)}
    X = design(theta); b, *_ = np.linalg.lstsq(X, y, rcond=None)
    return {'spec':'restricted','weights':weights,'coefs':b,'theta':theta,'names':names,'converged':bool(res.success)}

def predict_with_midas(fit, base_X, macro_list):
    base = np.asarray(base_X, float)
    cols = [macro_list[v] @ fit['weights'][v] for v in fit['names']]
    X = np.column_stack([np.ones(len(base)), base, np.column_stack(cols)])
    return X @ fit['coefs']

def fit_umidas(y, base_X, macro_list):
    names = list(macro_list)
    M = np.column_stack([macro_list[v] for v in names])
    X = np.column_stack([np.ones(len(y)), np.asarray(base_X, float), M])
    b, *_ = np.linalg.lstsq(X, np.asarray(y, float), rcond=None)
    return {'spec':'umidas','coefs':b,'names':names,'ncols':{v: macro_list[v].shape[1] for v in names}}

def predict_umidas(fit, base_X, macro_list):
    M = np.column_stack([macro_list[v] for v in fit['names']])
    X = np.column_stack([np.ones(len(np.asarray(base_X))), np.asarray(base_X, float), M])
    return X @ fit['coefs']

def make_mask(y, base_X, mlist):
    m = ~np.isnan(y) & ~np.isnan(np.asarray(base_X, float)).any(axis=1)
    for v in mlist: m = m & ~np.isnan(mlist[v]).any(axis=1)
    return m


## 6. Two-stage protocol — spec selection, then walk-forward ladder
**Stage 1** picks the weight family (U-MIDAS / Beta / Almon) by val WDA. **Stage 2** walk-forwards
(expanding, refit every 4 weeks). Ladder now includes the public-info rungs `EXOG+FRED_daily`,
`EXOG+COT`, `EXOG+FRED_daily+COT`, with FRED+COT folded into `EXOG+ALL`.


In [ ]:
y_all    = all_w['silver_return'].values
base_all = all_w[EXOG_LAGS].values
idx      = np.arange(len(all_w)); n_train_only = len(train_w)
sub      = lambda rm, ml: {v: ml[v][rm] for v in ml}

mask  = make_mask(y_all, base_all, macro_lags)
m_tr1 = mask & (idx <  n_train_only)
m_v   = mask & (idx >= n_train_only) & (idx < n_train)
kk = len(macro_lags)
fits1 = {
  'U-MIDAS':     fit_umidas    (y_all[m_tr1], base_all[m_tr1], sub(m_tr1, macro_lags)),
  'Beta-MIDAS':  fit_with_midas(y_all[m_tr1], base_all[m_tr1], sub(m_tr1, macro_lags), nbeta_w,  np.tile([1.,5.],kk), np.full(2*kk, 0.1)),
  'Almon-MIDAS': fit_with_midas(y_all[m_tr1], base_all[m_tr1], sub(m_tr1, macro_lags), nealmon_w, np.tile([0.,0.],kk), np.full(2*kk, -5.0)),
}
print('Stage 1 — val scores:')
s1 = []
for nm, ft in fits1.items():
    pv = (predict_umidas if ft['spec']=='umidas' else predict_with_midas)(ft, base_all[m_v], sub(m_v, macro_lags))
    s1.append(evaluate(nm, y_all[m_v], pv))
best_spec = max(s1, key=lambda r: r['wda'])['model']
print('=> winner by val WDA:', best_spec)
pd.DataFrame(s1).to_csv(DATA+'midas_stage1_specs.csv', index=False)

# ── Stage 2 ──────────────────────────────────────────────────────────────────
RETRAIN_EVERY = 4
base_F  = base_all[mask]; gs_F = all_w[GS_LAGS].values[mask]
fred_F  = all_w[FRED_LAGS].values[mask]; cot_F = all_w[COT_LAGS].values[mask]
macro_F = sub(mask, macro_lags); dates_F = all_w.index[mask]
orig    = idx[mask]; test_pos = np.where(orig >= n_train)[0]
y_F     = y_all[mask]; y_te = y_F[test_pos]; dates_te = dates_F[test_pos]
if sentiment_available:
    r_F = all_w['reddit_lag1'].values[mask]; n_F = all_w['news_lag1'].values[mask]

def fit_dispatch(y, baseX, mlist):
    k = len(mlist)
    if best_spec == 'U-MIDAS':    return fit_umidas(y, baseX, mlist)
    if best_spec == 'Beta-MIDAS': return fit_with_midas(y, baseX, mlist, nbeta_w,  np.tile([1.,5.],k), np.full(2*k, 0.1))
    return fit_with_midas(y, baseX, mlist, nealmon_w, np.tile([0.,0.],k), np.full(2*k, -5.0))
def pred_dispatch(fit, baseX, mlist):
    return (predict_umidas if fit['spec']=='umidas' else predict_with_midas)(fit, baseX, mlist)

def walk_forward(baseX, use_midas):
    preds = np.full(len(test_pos), np.nan); fit = None
    for j, p in enumerate(test_pos):
        if j % RETRAIN_EVERY == 0:
            if use_midas:
                fit = fit_dispatch(y_F[:p], baseX[:p], {v: macro_F[v][:p] for v in macro_F})
            else:
                X = np.column_stack([np.ones(p), baseX[:p]]); fit, *_ = np.linalg.lstsq(X, y_F[:p], rcond=None)
        if use_midas:
            preds[j] = pred_dispatch(fit, baseX[p:p+1], {v: macro_F[v][p:p+1] for v in macro_F})[0]
        else:
            preds[j] = np.concatenate([[1.0], baseX[p]]) @ fit
    return preds

all_preds = {}; all_results = []
def run_variant(label, baseX, use_midas):
    all_preds[label] = walk_forward(baseX, use_midas)
    all_results.append(evaluate(label, y_te, all_preds[label]))

all_preds['Naive (t-1 week)'] = np.concatenate([[np.nan], y_te[:-1]])
all_results.append(evaluate('Naive (t-1 week)', y_te, all_preds['Naive (t-1 week)']))
all_preds['Drift (prevailing mean)'] = np.array([y_F[:p].mean() for p in test_pos])
all_results.append(evaluate('Drift (prevailing mean)', y_te, all_preds['Drift (prevailing mean)']))

run_variant('EXOG',       base_F, False)
run_variant('EXOG+Macro', base_F, True)
if len(GS_LAGS):
    run_variant('EXOG+GS', np.column_stack([base_F, gs_F]), False)
# public-info rungs (mirror 01_arima): FRED_daily / COT / both
run_variant('EXOG+FRED_daily',     np.column_stack([base_F, fred_F]),        False)
run_variant('EXOG+COT',            np.column_stack([base_F, cot_F]),         False)
run_variant('EXOG+FRED_daily+COT', np.column_stack([base_F, fred_F, cot_F]), False)
if sentiment_available:
    run_variant('EXOG+Reddit',          np.column_stack([base_F, r_F]),      False)
    run_variant('EXOG+News',            np.column_stack([base_F, n_F]),      False)
    run_variant('EXOG+Reddit+News',     np.column_stack([base_F, r_F, n_F]), False)
    run_variant('EXOG+Macro+Sentiment', np.column_stack([base_F, r_F, n_F]), True)
    if len(GS_LAGS):
        run_variant('EXOG+GS+Sentiment', np.column_stack([base_F, gs_F, r_F, n_F]), False)
        # kitchen sink = GS + FRED_daily + COT + Sentiment + Macro
        run_variant('EXOG+ALL', np.column_stack([base_F, gs_F, fred_F, cot_F, r_F, n_F]), True)

L = test_pos[-1] + 1
fit_em = fit_dispatch(y_F[:L], base_F[:L], {v: macro_F[v][:L] for v in macro_F})


## 7. Results table

In [ ]:
metrics_df = pd.DataFrame(all_results)
drift_p = all_preds['Drift (prevailing mean)']
metrics_df['oos_r2'] = [oos_r2(y_te, all_preds[m], drift_p) for m in metrics_df['model']]
metrics_df.to_csv(DATA+'metrics_midas.csv', index=False)
print(metrics_df.round(6).to_string(index=False))


### Always-up directional benchmark

In [ ]:
# Always-up directional line — the Drift's sign is constant-positive, so its WDA = the
# magnitude-weighted up-share. The per-period benchmark any directional model must beat.
print('Always-up WDA by period (directional benchmark to beat):')
_s = pd.Series(y_te, index=dates_te)
for lbl, (a, b) in PERIODS.items():
    sub_ = _s.loc[a:b]
    if len(sub_) < 4: continue
    au = np.sum(np.abs(sub_.values) * (sub_.values > 0)) / np.sum(np.abs(sub_.values))
    print(f'  {lbl:22s} always-up WDA = {au:.3f}  (n={len(sub_)})')


## 8. Fitted macro lag-weight profiles (EXOG+Macro)

In [ ]:
if fit_em['spec'] == 'restricted':
    ws = fit_em['weights']; vs = list(ws); ncol = 2; nrow = int(np.ceil(len(vs)/ncol))
    fig, axes = plt.subplots(nrow, ncol, figsize=(10, 3*nrow)); axes = np.atleast_1d(axes).ravel()
    for ax, v in zip(axes, vs):
        w = ws[v]; ax.plot(range(1, len(w)+1), w, '-o', color='steelblue')
        ax.set_title(v); ax.set_xlabel('monthly lag (1=recent)'); ax.set_ylabel('weight')
    for ax in axes[len(vs):]: ax.axis('off')
    fig.suptitle(f'EXOG+Macro: fitted {best_spec} lag-weight profiles', fontweight='bold')
    plt.tight_layout(); plt.show()
else:
    nb = 1 + base_F.shape[1]; mc = fit_em['coefs'][nb:]
    names = [f'{v}_mlag{k}' for v in fit_em['names'] for k in range(1, fit_em['ncols'][v]+1)]
    print('U-MIDAS macro coefficients:', dict(zip(names, np.round(mc, 4))))


## 9. Period breakdown — best by WDA *and* by RMSE

In [ ]:
cand = metrics_df[~metrics_df['model'].isin(['Naive (t-1 week)','Drift (prevailing mean)'])]
best_wda  = cand.loc[cand['wda'].idxmax(),  'model']
best_rmse = cand.loc[cand['rmse'].idxmin(), 'model']
print(f'Best by WDA: {best_wda}   |   Best by RMSE: {best_rmse}')
for tag, bn, fper, fpred in [('WDA',  best_wda,  'period_midas_weekly.csv',      'preds_midas_best_weekly.csv'),
                             ('RMSE', best_rmse, 'period_midas_rmse_weekly.csv', 'preds_midas_bestrmse_weekly.csv')]:
    bp = all_preds[bn]
    pp = period_metrics(np.asarray(y_te), np.asarray(bp), dates_te, PERIODS)
    pp.to_csv(DATA+fper)
    pd.DataFrame({'Date': dates_te, 'actual': y_te, 'predicted': bp}).to_csv(DATA+fpred, index=False)
    print(f'\n[{tag}-best = {bn}]'); print(pp.round(3))
best_name, best_pred = best_wda, all_preds[best_wda]   # WDA-best drives per-period PT + the zoom default


## 10. Significance tests
Primary **DM-vs-Drift floor** (se + ae) + **OOS R²**; secondary incremental DM vs `EXOG`,
Pesaran-Timmermann (per-variant + multiple-testing summary + per-period for the best variant).


In [ ]:
drift_p = all_preds['Drift (prevailing mean)']
print('Floor test — each variant vs Drift   [primary: efficiency arbiter]'); print('-'*92)
for nm in all_preds:
    if nm in ('Drift (prevailing mean)','Naive (t-1 week)'): continue
    print(f'{nm:26s}  OOS_R2={oos_r2(y_te, all_preds[nm], drift_p):+.4f}')
    diebold_mariano(y_te, all_preds[nm], drift_p, nm, 'Drift', loss='se')
    diebold_mariano(y_te, all_preds[nm], drift_p, nm, 'Drift', loss='ae')
print('\nIncremental test — each variant vs EXOG   [secondary]'); print('-'*92)
for nm in all_preds:
    if nm in ('EXOG','Naive (t-1 week)','Drift (prevailing mean)'): continue
    diebold_mariano(y_te, all_preds['EXOG'], all_preds[nm], 'EXOG', nm, loss='se')
print('\nPesaran-Timmermann — directional   [secondary]'); print('-'*92)
for nm in all_preds:
    if nm == 'Naive (t-1 week)': continue
    pesaran_timmermann(y_te, all_preds[nm], name=nm)

# Multiple-testing summary + per-period PT for the best variant (selection-bias aware)
pts = {nm: pesaran_timmermann(y_te, all_preds[nm]) for nm in all_preds
       if nm not in ('Naive (t-1 week)','Drift (prevailing mean)')}
sig = [nm for nm, r in pts.items() if r and r.get('p') is not None and np.isfinite(r['p']) and r['p'] < 0.05]
nt = len(pts)
print(f'\nPT multiple-testing: {len(sig)}/{nt} significant at p<0.05  (~{0.05*nt:.1f} expected by chance)')
if sig: print('  significant:', sig)
print(f'  (best-by-WDA is the max over ~{nt} ablations — single-variant PT hits are selection-biased)')
print(f'\nPer-period PT — best variant ({best_name}):')
_s = pd.Series(y_te, index=dates_te); _p = pd.Series(best_pred, index=dates_te)
for lbl, (a, b) in PERIODS.items():
    yy = _s.loc[a:b].values; pp = _p.loc[a:b].values
    if len(yy) < 12: continue
    pesaran_timmermann(yy, pp, name=f'  {lbl}')


## 11. 2026 zoom — WDA-best and RMSE-best (vs Drift)

In [ ]:
m26 = dates_te >= pd.Timestamp('2026-01-01')
if m26.any():
    drift_p = all_preds['Drift (prevailing mean)']
    fig, axes = plt.subplots(2, 1, figsize=(10, 7), sharex=True)
    for ax, (tag, bn) in zip(axes, [('WDA', best_wda), ('RMSE', best_rmse)]):
        bp = np.asarray(all_preds[bn])
        ax.axhline(0, color='grey', ls=':', lw=.5)
        ax.plot(dates_te[m26], np.asarray(y_te)[m26], 'k-o', label='actual')
        ax.plot(dates_te[m26], np.asarray(drift_p)[m26], ':', color='grey', label='Drift')
        ax.plot(dates_te[m26], bp[m26], '--o', color='#c0392b', label=bn)
        ax.set_title(f'2026 — {tag}-best: {bn}'); ax.set_ylabel('weekly log-return'); ax.legend(fontsize=8)
    plt.tight_layout(); plt.show()
else:
    print('No 2026 data in test set yet.')
